<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/Excercise_Segmented_mean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.05-Serial-vs-Parallel/Sources/ach.h -nv -O Sources/ach.h

2026-09-26 15:37:04 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.05-Serial-vs-Parallel/Sources/ach.h [4942/4942] -> "Sources/ach.h" [1]


In [22]:
%%writefile Sources/segmented_mean.cpp
#include "ach.h"

// functor for mean:
struct functor_mean{
  int width;
  __host__ __device__ float operator()(float value)
  {
    return value/width;
  }
};

// function for getting means from a given temp:
thrust::universal_vector<float> row_temperatures(
  int height, int width, thrust::universal_vector<int> &row_ids, thrust::universal_vector<float> &temp)
{

  // store to a thrust vector called means:
  thrust::universal_vector<float> means(height);
  // row_ids
  auto row_ids_begin = thrust::make_transform_iterator(
    thrust::make_counting_iterator(0),
    [width] __host__ __device__(int id)
    {
      return id/width;
    }
  );

  auto row_ids_end = row_ids_begin + temp.size();

  auto means_output_it = thrust::make_transform_output_iterator(
    means.begin(),
    functor_mean{width}
  );
  // reduce_by_key
  thrust::reduce_by_key(
    thrust::device,
    row_ids_begin,
    row_ids_end,
    temp.begin(),
    thrust::make_discard_iterator(),
    means_output_it
  );

  return means;
}


Overwriting Sources/segmented_mean.cpp


In [23]:
!nvcc --extended-lambda -o /tmp/a.out Sources/segmented_mean.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

computed in 0.00542626 s
achieved throughput: 184.289 GB/s
maximal bandwidth: 298.083 GB/s
row 0: { 90, 90, ..., 90 } = 90
row 1: { 15, 15, ..., 15 } = 15
row 2: { 15, 15, ..., 15 } = 15
